In [14]:
from itertools import permutations
import pandas as pd
from graphviz import Digraph
import os

In [15]:
ROLES = ["Commutes", "Populations", "Salaries"]

In [16]:
def specialist_patterns_for_team_size(team_size):
    """
    Return all distinct (n_commutes, n_populations, n_salaries) tuples
    for a given team size, following your rule:
      - team_size = 3: (1,1,1)
      - team_size = 4: all permutations of (2,1,1)
      - team_size = 5: all permutations of (2,2,1)
      - team_size = 6: (2,2,2)
    """
    if team_size == 3:
        base = (1, 1, 1)
        patterns = [base]
    elif team_size == 4:
        base = (2, 1, 1)
        patterns = set(permutations(base))
    elif team_size == 5:
        base = (2, 2, 1)
        patterns = set(permutations(base))
    elif team_size == 6:
        base = (2, 2, 2)
        patterns = [base]
    else:
        raise ValueError("Team size must be in {3,4,5,6} for this spec.")
    
    # Make ordering stable
    patterns = sorted(patterns)
    return patterns

In [17]:
def make_specialist_stimuli_for_size(team_size):
    """
    Generate all specialist stimuli for a given team size.
    Each stimulus is a dict with:
      - team_size
      - structure = "specialist"
      - allocation_pattern: {role: count}
      - stimulus_id: e.g., "S4_spec_0"
      - members: list of analysts with a single role each
    """
    patterns = specialist_patterns_for_team_size(team_size)
    stimuli = []

    for idx, counts in enumerate(patterns):
        # counts is a tuple (n_commutes, n_populations, n_salaries)
        allocation_pattern = dict(zip(ROLES, counts))

        members = []
        analyst_id = 1
        for role, n in allocation_pattern.items():
            for _ in range(n):
                members.append({
                    "analyst_id": analyst_id,
                    "name": f"Analyst {analyst_id}",
                    "role": role,             # specialists: exactly one role
                    "roles": [role],          # (for symmetry with generalists)
                })
                analyst_id += 1

        stimuli.append({
            "team_size": team_size,
            "structure": "specialist",
            "allocation_pattern": allocation_pattern,
            "stimulus_id": f"S{team_size}_spec_{idx}",
            "members": members,
        })

    return stimuli

In [18]:
def make_generalist_stimulus_for_size(team_size):
    """
    Generate the single generalist stimulus for a given team size.
    All analysts handle all three roles.
    """
    members = []
    for i in range(team_size):
        members.append({
            "analyst_id": i + 1,
            "name": f"Analyst {i+1}",
            "role": "Generalist",
            "roles": ROLES.copy(),   # each analyst does all 3 stages
        })

    stimulus = {
        "team_size": team_size,
        "structure": "generalist",
        "allocation_pattern": {role: team_size for role in ROLES},  # conceptually all can do all
        "stimulus_id": f"S{team_size}_gen",
        "members": members,
    }
    return stimulus

In [19]:
def generate_all_stimuli(team_sizes=(3,4,5,6)):
    """
    Generate all stimuli across team sizes and structures.
    Returns: list of stimulus dicts.
    """
    all_stimuli = []
    for s in team_sizes:
        # Generalist (one per team size)
        all_stimuli.append(make_generalist_stimulus_for_size(s))
        # Specialist (all patterns per team size)
        all_stimuli.extend(make_specialist_stimuli_for_size(s))
    return all_stimuli

In [20]:
def stimuli_to_summary_df(stimuli):
    """
    Convert stimuli list into a summary DataFrame:
      - one row per stimulus
      - columns: team_size, structure, stimulus_id, counts per role
    """
    rows = []
    for stim in stimuli:
        base = {
            "stimulus_id": stim["stimulus_id"],
            "team_size": stim["team_size"],
            "structure": stim["structure"],
        }
        # allocation_pattern always has counts per role
        alloc = stim["allocation_pattern"]
        for role in ROLES:
            base[f"n_{role.lower()}"] = alloc.get(role, 0)
        rows.append(base)
    return pd.DataFrame(rows).sort_values(["team_size", "structure", "stimulus_id"])

In [21]:
def render_org_chart(stimulus, outdir="org_charts", fmt="png"):
    """
    Render an org chart (specialist or generalist) to a PNG or PDF using graphviz.
    """
    os.makedirs(outdir, exist_ok=True)

    team_size = stimulus["team_size"]
    structure = stimulus["structure"]
    stim_id = stimulus["stimulus_id"]

    g = Digraph(stim_id, format=fmt)
    g.attr(rankdir="TB")  # vertical chart

    # Top label node
    g.node("team", 
           f"{structure.title()} Team (n={team_size})\n{stim_id}",
           shape="box", style="filled", fillcolor="lightgray")

    # Specialist structure: group by roles
    if structure == "specialist":
        # Role cluster nodes
        for role in ROLES:
            g.node(role, role, shape="box", style="filled", fillcolor="#E8F0FE")
            g.edge("team", role)
        
        # Add analyst nodes under each role
        for m in stimulus["members"]:
            node_id = f"{stim_id}_{m['analyst_id']}"
            g.node(node_id, m["name"], shape="ellipse")
            g.edge(m["role"], node_id)

    # Generalist structure: all analysts under team node
    else:
        for m in stimulus["members"]:
            node_id = f"{stim_id}_{m['analyst_id']}"
            g.node(node_id, m["name"], shape="ellipse")
            g.edge("team", node_id)

    # Save file
    outpath = os.path.join(outdir, stim_id)
    g.render(outpath, cleanup=True)

    return f"{outpath}.{fmt}"

In [22]:
stimuli = generate_all_stimuli(team_sizes=[3,4,5,6])

paths = []
for stim in stimuli:
    p = render_org_chart(stim, fmt="png")   # or fmt="pdf"
    paths.append(p)

paths[:10]

['org_charts/S3_gen.png',
 'org_charts/S3_spec_0.png',
 'org_charts/S4_gen.png',
 'org_charts/S4_spec_0.png',
 'org_charts/S4_spec_1.png',
 'org_charts/S4_spec_2.png',
 'org_charts/S5_gen.png',
 'org_charts/S5_spec_0.png',
 'org_charts/S5_spec_1.png',
 'org_charts/S5_spec_2.png']